# Async Execution in CrewAI

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) CrewAI roadmap.

You will use task-level `async_execution`, `context` for join semantics, and crew-level `akickoff()` with `asyncio.gather`.

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Async tasks with `context` and `akickoff()`

Tasks with `async_execution=True` run concurrently where the process allows. A downstream task that lists them in `context` waits for **all** of them, then runs with their outputs.

The second code cell below runs **two crews in parallel** via `asyncio.gather` and `akickoff()`.

In [ ]:
import asyncio

from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Produce a short factual note from the prompt only",
    backstory="You are concise and avoid inventing sources.",
    verbose=True,
)
analyst = Agent(
    role="Market Analyst",
    goal="Summarize market angles briefly from the prompt",
    backstory="You focus on clarity over length.",
    verbose=True,
)
writer = Agent(
    role="Writer",
    goal="Merge upstream task outputs into one short brief",
    backstory="You synthesize without adding new claims.",
    verbose=True,
)

task_1 = Task(
    description="Research AI trends (reasoning only; no web).",
    expected_output="Trend report: 3 bullet points.",
    agent=researcher,
    async_execution=True,
)
task_2 = Task(
    description="Research market data themes (reasoning only; no web).",
    expected_output="Market report: 3 bullet points.",
    agent=analyst,
    async_execution=True,
)
synthesis = Task(
    description="Synthesize the trend and market reports into one combined note.",
    expected_output="Combined report under 200 words.",
    agent=writer,
    context=[task_1, task_2],
)

crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_1, task_2, synthesis],
    process=Process.sequential,
    verbose=True,
)

out = await crew.akickoff()
print(out)

In [ ]:
runner_a = Agent(
    role="Topic summarizer A",
    goal="One paragraph on the given topic",
    backstory="Stay on topic and brief.",
    verbose=True,
)
runner_b = Agent(
    role="Topic summarizer B",
    goal="One paragraph on the given topic",
    backstory="Stay on topic and brief.",
    verbose=True,
)

t_a = Task(
    description="Summarize the topic {topic} in one paragraph.",
    expected_output="One paragraph.",
    agent=runner_a,
)
t_b = Task(
    description="Summarize the topic {topic} in one paragraph.",
    expected_output="One paragraph.",
    agent=runner_b,
)

crew_ai = Crew(agents=[runner_a], tasks=[t_a], process=Process.sequential, verbose=True)
crew_ml = Crew(agents=[runner_b], tasks=[t_b], process=Process.sequential, verbose=True)

results = await asyncio.gather(
    crew_ai.akickoff(inputs={"topic": "AI"}),
    crew_ml.akickoff(inputs={"topic": "ML"}),
)
for i, r in enumerate(results):
    print(f"--- crew {i + 1} ---")
    print(r)

## Key takeaways

- **`async_execution=True`** lets independent tasks run concurrently; **`context=[...]`** on a later task waits for **all** listed tasks and merges their outputs.
- **`akickoff()`** is the native async entrypoint for a crew; pair it with **`asyncio.gather`** to run **multiple crews** at once.
- Use async for **parallel branches**, **batches**, and **multi-crew** workloads; keep dependencies explicit with **`context`**.
